In [4]:
import importlib.util
import platform
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

print(f"Python: {sys.version.split()[0]}")
print(f"Ejecutable: {sys.executable}")
print(f"Sistema: {platform.platform()}")
print(f"Entorno: {'Google Colab' if IN_COLAB else 'local'}")

for module_name, package_name in {
    "torch": "PyTorch",
    "torchaudio": "torchaudio",
    "fish_speech": "Fish Speech",
}.items():
    status = "disponible" if importlib.util.find_spec(module_name) else "FALTA"
    print(f"{package_name}: {status}")

if importlib.util.find_spec("torch") is not None:
    import torch
    CUDA_AVAILABLE = torch.cuda.is_available()
    DEVICE = "cuda" if CUDA_AVAILABLE else "cpu"
    print(f"CUDA disponible: {CUDA_AVAILABLE}")
    if CUDA_AVAILABLE:
        print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    CUDA_AVAILABLE = False
    DEVICE = "cpu"

WORKSPACE_DIR = Path("/content") if IN_COLAB else Path.cwd()
INPUT_AUDIO = WORKSPACE_DIR / "input.wav"
OUTPUT_AUDIO = WORKSPACE_DIR / "output_s2s.wav"
LANGUAGE = "es"

print(f"Dispositivo seleccionado: {DEVICE}")
print(f"Audio de entrada: {INPUT_AUDIO}")
print(f"Audio de salida: {OUTPUT_AUDIO}")

Python: 3.12.10
Ejecutable: c:\Users\oitav\Documents\VIU\TFM\voice-agents\voiceagent\Scripts\python.exe
Sistema: Windows-11-10.0.26200-SP0
Entorno: local
PyTorch: disponible
torchaudio: disponible
Fish Speech: disponible
CUDA disponible: False
Dispositivo seleccionado: cpu
Audio de entrada: c:\Users\oitav\Documents\VIU\TFM\voice-agents\speech-to-speech\input.wav
Audio de salida: c:\Users\oitav\Documents\VIU\TFM\voice-agents\speech-to-speech\output_s2s.wav


# Agente Speech-to-Speech con Fish Speech S2 Pro

Notebook incremental para validar un prototipo S2S por etapas:

```text
Audio WAV -> Fish Speech S2 Pro -> Audio WAV
```

El objetivo inicial es comprobar la inferencia con un archivo de audio. El streaming, el microfono en tiempo real, las interrupciones y la evaluacion comparativa se añadiran despues de validar este flujo basico.

## 1. Instalacion y dependencias

Ejecuta la instalacion desde una terminal con el entorno `voiceagent` activado. No se ejecuta automaticamente porque Fish Speech puede requerir versiones y recursos especificos.

In [6]:
import importlib.util
import subprocess
import sys

INSTALL_ON_COLAB = True

if importlib.util.find_spec("fish_speech") is None and IN_COLAB and INSTALL_ON_COLAB:
    print("Instalando Fish Speech en Google Colab...")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "git+https://github.com/fishaudio/fish-speech.git",
        ],
        check=True,
    )
    print("Instalacion terminada. Reinicia el entorno de ejecucion y vuelve a ejecutar desde la celda 1.")

if importlib.util.find_spec("fish_speech") is None:
    raise ImportError(
        "Falta Fish Speech. En local, ejecuta la instalacion desde PowerShell. "
        "En Colab, deja INSTALL_ON_COLAB=True y ejecuta de nuevo esta celda."
    )

import fish_speech
from importlib.metadata import version

FISH_SPEECH_VERSION = version("fish-speech")
FISH_SPEECH_S2_SUPPORTED = False

print(f"Fish Speech instalado: {FISH_SPEECH_VERSION}")
print("API S2Model disponible: False")
print(
    "La version instalada incluye modelos DAC y text2semantic, "
    "pero no fish_speech.models.s2 ni fish_speech.utils.audio."
)
print("Este notebook necesita una implementacion S2S compatible antes de continuar.")

Fish Speech instalado: 2.0.0
API S2Model disponible: False
La version instalada incluye modelos DAC y text2semantic, pero no fish_speech.models.s2 ni fish_speech.utils.audio.
Este notebook necesita una implementacion S2S compatible antes de continuar.


## 2. Configuracion del experimento

Fija el modelo, el idioma y los archivos antes de cargar pesos. La primera prueba usa un WAV local para poder repetirla sin depender del microfono.

In [ ]:
from pathlib import Path

MODEL_NAME = "fishaudio/fish-speech-s2-pro"
INPUT_AUDIO = Path("input.wav")
OUTPUT_AUDIO = Path("output_s2s.wav")
LANGUAGE = "es"
MAX_LENGTH_SECONDS = 12.0
TEMPERATURE = 0.7
TOP_P = 0.9

print(f"Modelo: {MODEL_NAME}")
print(f"Idioma: {LANGUAGE}")
print(f"Entrada: {INPUT_AUDIO.resolve()}")
print(f"Salida: {OUTPUT_AUDIO.resolve()}")

## 2.1. Cargar un audio en Google Colab

En Colab el microfono pertenece al navegador, no al sistema donde se ejecuta Python. Para esta primera prueba, sube un WAV desde el navegador. En local esta celda no hace nada.

In [ ]:
if IN_COLAB:
    from google.colab import files

    uploaded_files = files.upload()
    if not uploaded_files:
        raise RuntimeError("No se ha seleccionado ningun archivo WAV.")

    uploaded_name = next(iter(uploaded_files))
    uploaded_path = Path(uploaded_name)
    if uploaded_path.suffix.lower() != ".wav":
        raise ValueError("Selecciona un archivo con extension .wav.")

    INPUT_AUDIO = WORKSPACE_DIR / uploaded_path.name
    print(f"Audio cargado en Colab: {INPUT_AUDIO}")
else:
    print("Entorno local: se conserva la ruta configurada para INPUT_AUDIO.")

## 3. Validar el audio de entrada

Comprueba primero que el archivo existe, se puede leer y tiene una duracion razonable. Esta celda no carga ningun modelo.

In [ ]:
import soundfile as sf

if not INPUT_AUDIO.exists():
    raise FileNotFoundError(
        f"No existe el audio de entrada: {INPUT_AUDIO.resolve()}. "
        "Coloca un WAV en la carpeta del notebook o cambia INPUT_AUDIO."
    )

audio_info = sf.info(INPUT_AUDIO)
print(f"Formato: {audio_info.format}")
print(f"Frecuencia: {audio_info.samplerate} Hz")
print(f"Canales: {audio_info.channels}")
print(f"Duracion: {audio_info.duration:.2f} s")

if audio_info.duration <= 0:
    raise ValueError("El audio de entrada esta vacio.")

## 4. Cargar audio y modelo

Esta celda prepara el audio en el dispositivo seleccionado y carga los pesos. La descarga puede ser grande y la primera carga puede tardar.

In [ ]:
import time
import torch

if not FISH_SPEECH_S2_SUPPORTED:
    raise RuntimeError(
        "La version instalada de Fish Speech no expone una API S2S compatible "
        "con este prototipo. No ejecutes esta celda hasta elegir una version "
        "o implementacion que proporcione carga de audio y generacion audio-audio."
    )

if "S2Model" not in globals():
    raise RuntimeError(
        "S2Model no esta disponible. Ejecuta primero la celda de dependencias "
        "despues de instalar una implementacion S2S compatible."
    )

load_started = time.perf_counter()
waveform, sample_rate = load_audio(str(INPUT_AUDIO))
waveform = waveform.to(DEVICE)

model = S2Model.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)
model.eval()

print(f"Audio cargado: {sample_rate} Hz")
print(f"Forma del audio: {tuple(waveform.shape)}")
print(f"Modelo cargado en: {DEVICE}")
print(f"Tiempo de carga: {time.perf_counter() - load_started:.2f} s")

## 5. Inspeccionar la API de inferencia

Antes de generar audio, comprueba que la versión instalada expone el método esperado. Si no existe, consulta la API de la versión instalada en lugar de continuar con una llamada incompatible.

In [ ]:
import inspect

inference_method = getattr(model, "generate_speech_to_speech", None)
print(f"Clase del modelo: {type(model).__name__}")
print(f"Metodo esperado disponible: {callable(inference_method)}")

if not callable(inference_method):
    available_methods = [
        name for name in dir(model)
        if "generate" in name.lower() or "speech" in name.lower()
    ]
    print("Metodos relacionados disponibles:", available_methods)
else:
    print("Firma detectada:")
    print(inspect.signature(inference_method))

## 6. Ejecutar una inferencia

Genera una respuesta a partir del WAV validado. Esta celda puede consumir bastante memoria y tiempo, sobre todo en CPU.

In [ ]:
if not callable(inference_method):
    raise AttributeError(
        "La version instalada no expone generate_speech_to_speech. "
        "Revisa la firma y los metodos mostrados en la celda anterior."
    )

inference_started = time.perf_counter()
with torch.inference_mode():
    output_waveform = inference_method(
        audio=waveform,
        sample_rate=sample_rate,
        language=LANGUAGE,
        max_length=MAX_LENGTH_SECONDS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
    )

inference_seconds = time.perf_counter() - inference_started
print(f"Inferencia completada en: {inference_seconds:.2f} s")
print(f"Tipo de salida: {type(output_waveform).__name__}")

## 7. Guardar y escuchar la respuesta

Guarda el resultado en WAV y muestra un reproductor en el notebook. La forma exacta de la salida puede variar según la versión de Fish Speech.

In [ ]:
from IPython.display import Audio, display

OUTPUT_AUDIO.parent.mkdir(parents=True, exist_ok=True)

if not isinstance(output_waveform, torch.Tensor):
    raise TypeError(
        "La salida no es un tensor de PyTorch. Ajusta esta celda al formato "
        "devuelto por la version instalada antes de guardarla."
    )

save_audio(str(OUTPUT_AUDIO), output_waveform.detach().cpu(), sample_rate)
print(f"Audio guardado en: {OUTPUT_AUDIO.resolve()}")
display(Audio(filename=str(OUTPUT_AUDIO)))

## 8. Registrar una primera medicion

Esta medicion sirve como referencia exploratoria. Para el experimento final sera necesario repetir cada prueba despues de un calentamiento y guardar tambien modelo, revision, hardware y versiones.

In [ ]:
import json
from datetime import datetime, timezone

measurement = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "model": MODEL_NAME,
    "language": LANGUAGE,
    "device": DEVICE,
    "input_audio": str(INPUT_AUDIO),
    "input_duration_seconds": audio_info.duration,
    "sample_rate": sample_rate,
    "inference_seconds": inference_seconds,
}

print(json.dumps(measurement, indent=2, ensure_ascii=False))

## 9. Siguientes ampliaciones

1. Validar la API y la licencia de la version exacta de Fish Speech.
2. Añadir grabacion desde microfono y reproduccion local.
3. Repetir la prueba con varias frases en español.
4. Medir latencia hasta el primer audio, tiempo total y consumo de recursos.
5. Incorporar VAD, streaming e interrupciones solo despues de validar la inferencia por WAV.
6. Comparar los resultados con la linea base `faster-whisper -> Ollama -> Piper`.

Este notebook debe considerarse un prototipo experimental hasta completar esas comprobaciones.